# 1.2 数据清洗 (Data Cleaning)

数据清洗是提升训练数据质量的关键环节，直接影响模型的最终性能。

本节涵盖：
- 质量过滤
- 去重
- PII去除
- 有害内容过滤
- 格式标准化

## 1. 质量过滤

**目的**：去除低质量、噪声数据

**基本原理**：训练轻量级分类器（如fastText）对文本质量打分，过滤包含大量重复、乱码、模板化内容的低质量文档；使用困惑度过滤（用已有模型计算PPL，过高或过低的文本被过滤）。

**常用质量指标**：
- 文本长度（过短通常质量低）
- 重复行比例（高重复=模板/爬取错误）
- 特殊字符比例（过高=乱码）
- 语言模型困惑度（过高=不自然，过低=模板化）
- 词汇多样性（低多样性=重复内容）

In [ ]:
import re
from collections import Counter

class QualityFilter:
    def __init__(self, min_length=50, max_length=100000,
                 max_repeat_ratio=0.3, max_special_char_ratio=0.2,
                 min_unique_token_ratio=0.1):
        self.min_length = min_length
        self.max_length = max_length
        self.max_repeat_ratio = max_repeat_ratio
        self.max_special_char_ratio = max_special_char_ratio
        self.min_unique_token_ratio = min_unique_token_ratio

    def compute_repeat_ratio(self, text):
        lines = text.split('\n')
        if len(lines) <= 1:
            return 0.0
        line_counts = Counter(lines)
        repeated = sum(count - 1 for count in line_counts.values() if count > 1)
        return repeated / len(lines)

    def compute_special_char_ratio(self, text):
        if not text:
            return 0.0
        special = sum(1 for c in text if not c.isalnum() and c not in ' \n\t.,!?;:\'"()-')
        return special / len(text)

    def compute_unique_token_ratio(self, text):
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        return len(set(tokens)) / len(tokens)

    def filter(self, text):
        reasons = []
        if len(text) < self.min_length:
            reasons.append(f'too_short({len(text)}<{self.min_length})')
        if len(text) > self.max_length:
            reasons.append(f'too_long({len(text)}>{self.max_length})')

        repeat_ratio = self.compute_repeat_ratio(text)
        if repeat_ratio > self.max_repeat_ratio:
            reasons.append(f'high_repeat({repeat_ratio:.2f}>{self.max_repeat_ratio})')

        special_ratio = self.compute_special_char_ratio(text)
        if special_ratio > self.max_special_char_ratio:
            reasons.append(f'high_special({special_ratio:.2f}>{self.max_special_char_ratio})')

        unique_ratio = self.compute_unique_token_ratio(text)
        if unique_ratio < self.min_unique_token_ratio:
            reasons.append(f'low_diversity({unique_ratio:.2f}<{self.min_unique_token_ratio})')

        return len(reasons) == 0, reasons

filter_engine = QualityFilter(min_length=50, max_repeat_ratio=0.3,
                              max_special_char_ratio=0.2, min_unique_token_ratio=0.1)

test_docs = [
    ('Short text', 'Hello world'),
    ('High quality', 'Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data. These systems improve their performance over time without being explicitly programmed. Common applications include image recognition, natural language processing, and recommendation systems.'),
    ('Repetitive', 'Click here to buy\nClick here to buy\nClick here to buy\nClick here to buy\nClick here to buy\nClick here to buy\nClick here to buy\nClick here to buy\nClick here to buy'),
    ('Special chars', '\x00\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0a\x0b\x0c\x0d\x0e Some text with lots of control characters that make this document low quality for training purposes.'),
    ('Low diversity', 'the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the'),
]

print('=== Quality Filtering Demo ===')
for name, doc in test_docs:
    passed, reasons = filter_engine.filter(doc)
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {name}: {reasons if reasons else "all checks passed"}')

## 2. 去重

**目的**：消除冗余数据，避免模型记忆重复内容

**基本原理**：
- **精确去重**：对文档进行哈希比对，完全相同的文档只保留一份
- **模糊去重**：使用MinHash+LSH或SimHash算法计算文档相似度，将高度相似的文档进行去重处理

模糊去重是大模型数据清洗的核心步骤，因为互联网数据中存在大量近似重复内容（如同一新闻的不同转载、模板化页面等）。MinHash通过将文档的n-gram集合映射为多个哈希签名的最小值，在保持Jaccard相似度的同时大幅降低计算量。

In [ ]:
import hashlib
from collections import defaultdict

def get_shingles(text, k=5):
    tokens = text.lower().split()
    return set(' '.join(tokens[i:i+k]) for i in range(len(tokens) - k + 1))

def jaccard_similarity(set1, set2):
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

class MinHashDeduplicator:
    def __init__(self, num_hashes=128, threshold=0.8):
        self.num_hashes = num_hashes
        self.threshold = threshold
        self.hash_funcs = [
            lambda x, a=i, b=i*7+3: hash(f'{a}:{x}:{b}')
            for i in range(num_hashes)
        ]

    def compute_signature(self, text):
        shingles = get_shingles(text)
        if not shingles:
            return tuple([float('inf')] * self.num_hashes)
        signature = []
        for func in self.hash_funcs:
            min_hash = min(func(s) for s in shingles)
            signature.append(min_hash)
        return tuple(signature)

    def estimate_similarity(self, sig1, sig2):
        matches = sum(1 for a, b in zip(sig1, sig2) if a == b)
        return matches / self.num_hashes

    def deduplicate(self, documents):
        signatures = [(i, self.compute_signature(doc)) for i, doc in enumerate(documents)]
        keep = set(range(len(documents)))
        duplicates = []

        for i in range(len(signatures)):
            if signatures[i][0] not in keep:
                continue
            for j in range(i + 1, len(signatures))
                if signatures[j][0] not in keep:
                    continue
                sim = self.estimate_similarity(signatures[i][1], signatures[j][1])
                if sim >= self.threshold:
                    keep.discard(signatures[j][0])
                    duplicates.append((signatures[j][0], signatures[i][0], sim))

        return keep, duplicates

class ExactDeduplicator:
    def __init__(self):
        self.seen_hashes = set()

    def deduplicate(self, documents):
        keep = []
        duplicates = 0
        for doc in documents:
            doc_hash = hashlib.md5(doc.encode()).hexdigest()
            if doc_hash not in self.seen_hashes:
                self.seen_hashes.add(doc_hash)
                keep.append(doc)
            else:
                duplicates += 1
        return keep, duplicates

docs = [
    'The transformer architecture revolutionized natural language processing by introducing self attention mechanism',
    'The transformer architecture revolutionized natural language processing by introducing self attention mechanism',
    'The transformer architecture changed natural language processing by introducing the self attention mechanism',
    'Deep learning models require large amounts of training data to achieve good performance on downstream tasks',
    'Deep learning models need large amounts of training data to achieve good performance on downstream tasks',
    'Reinforcement learning from human feedback is a technique for aligning language models with human preferences',
]

print('=== Deduplication Demo ===')
print(f'Original documents: {len(docs)}')

exact_dedup = ExactDeduplicator()
exact_keep, exact_dupes = exact_dedup.deduplicate(docs)
print(f'\nExact dedup: {len(exact_keep)} kept, {exact_dupes} duplicates removed')

minhash_dedup = MinHashDeduplicator(num_hashes=128, threshold=0.7)
keep_indices, fuzzy_dupes = minhash_dedup.deduplicate(docs)
print(f'Fuzzy dedup (MinHash): {len(keep_indices)} kept')
for dup_idx, orig_idx, sim in fuzzy_dupes:
    print(f'  Doc {dup_idx} is duplicate of Doc {orig_idx} (similarity={sim:.2f})')
    print(f'    Original: {docs[orig_idx][:60]}...')
    print(f'    Duplicate: {docs[dup_idx][:60]}...')

## 3. PII去除

**目的**：保护个人隐私信息

**基本原理**：使用正则表达式和NER模型检测并移除个人身份信息（PII），如姓名、电话、邮箱、身份证号等敏感数据。这是数据合规（如GDPR、CCPA）的必要步骤。

**常见PII类型**：
- 邮箱地址
- 电话号码
- 身份证号/社保号
- 信用卡号
- IP地址
- 姓名（需NER模型）

In [ ]:
import re

class PIIRemover:
    def __init__(self):
        self.patterns = {
            'email': (r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[EMAIL]'),
            'phone_cn': (r'1[3-9]\d{9}', '[PHONE]'),
            'phone_us': (r'\(\d{3}\)\s*\d{3}-\d{4}|\d{3}-\d{3}-\d{4}', '[PHONE]'),
            'id_card_cn': (r'\d{17}[\dXx]', '[ID_CARD]'),
            'ssn_us': (r'\d{3}-\d{2}-\d{4}', '[SSN]'),
            'credit_card': (r'\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}', '[CREDIT_CARD]'),
            'ip_address': (r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '[IP]'),
        }

    def detect(self, text):
        findings = []
        for pii_type, (pattern, _) in self.patterns.items():
            matches = re.finditer(pattern, text)
            for match in matches:
                findings.append({
                    'type': pii_type,
                    'value': match.group(),
                    'start': match.start(),
                    'end': match.end(),
                })
        return findings

    def remove(self, text):
        findings = self.detect(text)
        redacted = text
        offset = 0
        for finding in sorted(findings, key=lambda x: x['start']):
            replacement = self.patterns[finding['type']][1]
            start = finding['start'] + offset
            end = finding['end'] + offset
            redacted = redacted[:start] + replacement + redacted[end:]
            offset += len(replacement) - (end - start)
        return redacted, findings

remover = PIIRemover()

test_texts = [
    'Please contact John at john.doe@example.com or call 13812345678 for assistance.',
    'My ID card number is 110101199001011234 and my credit card is 6222 0000 1234 5678.',
    'Server IP 192.168.1.100 is accessible at user@company.org with SSN 123-45-6789.',
    'This is a clean text without any personal information.',
]

print('=== PII Removal Demo ===')
for text in test_texts:
    redacted, findings = remover.remove(text)
    print(f'\nOriginal:  {text}')
    print(f'Redacted:  {redacted}')
    if findings:
        print(f'Detected:  {[(f["type"], f["value"]) for f in findings]}')
    else:
        print('Detected:  None')

## 4. 有害内容过滤

**目的**：移除有毒、偏见内容

**基本原理**：使用毒性分类器（如Perspective API）和偏见检测模型过滤包含仇恨言论、色情、暴力等有害内容的文档。产业实践中，通常使用基于关键词的快速过滤和基于模型的精细过滤两阶段方案。

In [ ]:
class ToxicityFilter:
    def __init__(self):
        self.toxic_keywords = {
            'hate_speech': ['hate', 'inferior race', 'subhuman'],
            'violence': ['kill', 'murder', 'bomb making'],
            'sexual': ['explicit content', 'nsfw'],
        }
        self.severity_weights = {
            'hate_speech': 1.0,
            'violence': 0.9,
            'sexual': 0.7,
        }

    def keyword_check(self, text):
        text_lower = text.lower()
        detected = {}
        for category, keywords in self.toxic_keywords.items():
            found = [kw for kw in keywords if kw in text_lower]
            if found:
                detected[category] = found
        return detected

    def compute_toxicity_score(self, text):
        detected = self.keyword_check(text)
        if not detected:
            return 0.0, detected
        score = sum(
            self.severity_weights[cat] * len(kws)
            for cat, kws in detected.items()
        )
        return min(score, 1.0), detected

    def filter(self, documents, threshold=0.3):
        clean = []
        filtered = []
        for doc in documents:
            score, detected = self.compute_toxicity_score(doc)
            if score >= threshold:
                filtered.append({'text': doc, 'score': score, 'reasons': detected})
            else:
                clean.append(doc)
        return clean, filtered

tox_filter = ToxicityFilter()

test_docs = [
    'Machine learning algorithms can be used for natural language processing tasks.',
    'This is a hate speech example that contains inferior race language.',
    'The research paper discusses bomb making techniques for historical analysis.',
    'Python is a popular programming language for data science and web development.',
    'Explicit content should be filtered out from training data.',
]

clean, filtered = tox_filter.filter(test_docs, threshold=0.3)

print('=== Toxicity Filtering Demo ===')
print(f'Clean documents: {len(clean)}/{len(test_docs)}')
print(f'Filtered documents: {len(filtered)}/{len(test_docs)}')

print('\n--- Filtered Documents ---')
for item in filtered:
    print(f'  Score: {item["score"]:.2f} | Reasons: {item["reasons"]}')
    print(f'  Text: {item["text"][:60]}...')

print('\n--- Clean Documents ---')
for doc in clean:
    print(f'  {doc[:60]}...')

## 5. 格式标准化

**目的**：统一数据格式以利于训练

**基本原理**：去除HTML标签、Markdown标记、异常Unicode字符，统一编码格式，规范化空白字符和换行符。格式标准化确保训练数据格式一致，避免模型学习到格式噪声。

In [ ]:
import re
import unicodedata

class FormatNormalizer:
    def __init__(self):
        self.html_tag_pattern = re.compile(r'<[^>]+>')
        self.multi_space_pattern = re.compile(r' {3,}')
        self.multi_newline_pattern = re.compile(r'\n{3,}')
        self.url_pattern = re.compile(r'https?://\S+')

    def remove_html_tags(self, text):
        text = re.sub(r'<script[^>]*>.*?</script>', '', text, flags=re.DOTALL)
        text = re.sub(r'<style[^>]*>.*?</style>', '', text, flags=re.DOTALL)
        text = self.html_tag_pattern.sub(' ', text)
        return text

    def normalize_unicode(self, text):
        text = unicodedata.normalize('NFKC', text)
        text = ''.join(c for c in text if unicodedata.category(c)[0] != 'C' or c in '\n\t')
        return text

    def normalize_whitespace(self, text):
        text = self.multi_newline_pattern.sub('\n\n', text)
        text = self.multi_space_pattern.sub(' ', text)
        text = '\n'.join(line.strip() for line in text.split('\n'))
        return text.strip()

    def normalize(self, text, keep_urls=True):
        text = self.remove_html_tags(text)
        text = self.normalize_unicode(text)
        if not keep_urls:
            text = self.url_pattern.sub('[URL]', text)
        text = self.normalize_whitespace(text)
        return text

normalizer = FormatNormalizer()

test_cases = [
    '<html><body><h1>Title</h1><p>Machine learning is <b>powerful</b>.</p><script>alert(1)</script></body></html>',
    'Too   many   spaces\n\n\n\nand newlines\n\n\n   here',
    'Unicode\u00a0non-breaking\u00a0spaces and\u200bzero-width\u200bchars and\x00null\x01bytes',
    'Visit https://example.com for more info and check https://github.com/repo',
]

print('=== Format Normalization Demo ===')
for i, text in enumerate(test_cases, 1):
    normalized = normalizer.normalize(text, keep_urls=True)
    normalized_no_url = normalizer.normalize(text, keep_urls=False)
    print(f'\n--- Case {i} ---')
    print(f'Input:     {repr(text[:80])}')
    print(f'Normalized: {normalized[:80]}')
    print(f'No URLs:    {normalized_no_url[:80]}')
    print(f'Length: {len(text)} -> {len(normalized)} (reduction: {(1-len(normalized)/len(text))*100:.1f}%)')